# NB22 — Is the divergence anything more than a threshold?

Three seeds have now been scored on the stress set (NB19, NB20) and on Ar-APT (NB21). The false-alarm effect reverses sign at seed 2 in both places, while the evasion cost against the strongest rewriter holds. The question is whether fusion changes the decision *boundary* rather than the decision *rule*: if the hybrid simply sits at a different operating point, and that point drifts between training runs, everything observed follows without any new mechanism.

**A. The matched operating point.** The neural threshold is moved until its false-alarm rate on the 400 untransformed Ar-APT human articles equals the hybrid's, and every comparison is repeated there. That anchor is used rather than the untransformed stress controls, whose rates sit near zero and cannot fix a threshold.

**B. Calibration drift.** The threshold each model needs for a fixed false-alarm rate is reported per seed. If the hybrid's threshold moves more across seeds than the neural one, the instability is calibration, and that is the finding.

**C. Which articles and which rewriters.** Whether the same articles drive the gap in every seed, and which rewriting model carries it.

CPU only, a few minutes, no model is changed.

In [ ]:
# ============================ 1/7  inputs ============================
import numpy as np, pandas as pd, itertools

P_SCORES = "/kaggle/input/notebooks/bahaaqassem/nb20-seed-aggregation/nb20_all_seed_scores.parquet"
P_ARAPT  = "/kaggle/input/notebooks/bahaaqassem/nb21-arapt-seeds/nb21_arapt_seed_scores.parquet"
N_BOOT = 2000
RNG = np.random.default_rng(42)

S = pd.read_parquet(P_SCORES)
need = ["p_hybrid_base", "p_hybrid_gen", "p_neural_base", "p_neural_gen", "task", "seed", "gen_id"]
assert not [c for c in need if c not in S.columns], "missing columns in the stress file"
print(f"[1/7] stress rows {len(S)} | per seed {S.groupby('seed').size().to_dict()}", flush=True)

A = pd.read_parquet(P_ARAPT)
assert {"p_hybrid", "p_neural", "kind", "seed"} <= set(A.columns), "missing columns in the Ar-APT file"
assert not A.p_hybrid.isna().any() and not A.p_neural.isna().any(), "Ar-APT scores contain gaps"
ANCHOR = A[A.kind == "human_orig"]
print(f"[1/7] Ar-APT rows {len(A)} | anchor (untransformed human) {len(ANCHOR)} "
      f"| per seed {ANCHOR.groupby('seed').size().to_dict()}", flush=True)
assert ANCHOR.groupby("seed").size().min() >= 200, "anchor too small to fix a threshold on"
seeds = sorted(int(s) for s in S.seed.unique())
print(f"[1/7] seeds present in both files: {seeds}", flush=True)

In [ ]:
# ============================ 2/7  B · how far each model's threshold drifts ============================
def rate(p, thr=0.5): return 100.0*float((np.asarray(p) >= thr).mean()) if len(p) else float("nan")

def threshold_for(scores, target):
    """threshold whose positive rate on `scores` is closest to `target` per cent"""
    grid = np.unique(np.concatenate([np.linspace(0.0, 1.0, 4001), np.asarray(scores)]))
    r = np.array([rate(scores, t) for t in grid])
    return float(grid[int(np.argmin(np.abs(r - target)))])

rows = []
for seed in seeds:
    a = ANCHOR[ANCHOR.seed == seed]
    row = {"seed": seed, "anchor_n": len(a),
           "FPR_hyb_at_05": round(rate(a.p_hybrid), 2), "FPR_neu_at_05": round(rate(a.p_neural), 2)}
    for target in (1.0, 5.0, 10.0):
        row[f"thr_hyb@{target:g}"] = round(threshold_for(a.p_hybrid.to_numpy(), target), 4)
        row[f"thr_neu@{target:g}"] = round(threshold_for(a.p_neural.to_numpy(), target), 4)
    rows.append(row)
D = pd.DataFrame(rows)
print("[2/7] operating points on the Ar-APT human anchor", flush=True)
print(D.to_string(index=False), flush=True)
print("", flush=True)
for target in (1.0, 5.0, 10.0):
    h = D[f"thr_hyb@{target:g}"]; n = D[f"thr_neu@{target:g}"]
    print(f"[2/7] threshold spread across seeds at a {target:g}% false-alarm target: "
          f"hybrid {h.max()-h.min():.4f}   neural {n.max()-n.min():.4f}", flush=True)
print("[2/7] a wider hybrid spread means fusion makes the decision boundary less stable across runs.", flush=True)

In [ ]:
# ============================ 3/7  A · match the neural threshold to the hybrid ============================
rows = []
THR = {}
for seed in seeds:
    a = ANCHOR[ANCHOR.seed == seed]
    target = rate(a.p_hybrid)                       # the hybrid's own false-alarm rate on the anchor
    thr = threshold_for(a.p_neural.to_numpy(), target)
    THR[seed] = thr
    s = S[S.seed == seed]; pol = s[s.task == "polish"]; hum = s[s.task == "humanize"]
    ap = A[(A.seed == seed) & (A.kind == "human_polished")]
    ai = A[(A.seed == seed) & (A.kind == "ai_generated")]
    rows.append({
        "seed": seed, "anchor_target": round(target, 2), "thr_neural": round(thr, 4),
        "anchor_neu_at_thr": round(rate(a.p_neural, thr), 2),
        "stress_FPR_hyb": round(rate(pol.p_hybrid_gen), 2),
        "stress_FPR_neu_05": round(rate(pol.p_neural_gen), 2),
        "stress_FPR_neu_matched": round(rate(pol.p_neural_gen, thr), 2),
        "stress_TPR_hyb": round(rate(hum.p_hybrid_gen), 2),
        "stress_TPR_neu_05": round(rate(hum.p_neural_gen), 2),
        "stress_TPR_neu_matched": round(rate(hum.p_neural_gen, thr), 2),
        "arapt_POL_hyb": round(rate(ap.p_hybrid), 2),
        "arapt_POL_neu_05": round(rate(ap.p_neural), 2),
        "arapt_POL_neu_matched": round(rate(ap.p_neural, thr), 2),
        "arapt_AI_hyb": round(rate(ai.p_hybrid), 2),
        "arapt_AI_neu_05": round(rate(ai.p_neural), 2),
        "arapt_AI_neu_matched": round(rate(ai.p_neural, thr), 2)})
M = pd.DataFrame(rows)
for a_, b_, name in (("stress_FPR_hyb", "stress_FPR_neu", "stress_FPR"),
                     ("stress_TPR_hyb", "stress_TPR_neu", "stress_TPR"),
                     ("arapt_POL_hyb",  "arapt_POL_neu",  "arapt_POL"),
                     ("arapt_AI_hyb",   "arapt_AI_neu",   "arapt_AI")):
    M[f"gap_{name}_05"]      = (M[a_] - M[f"{b_}_05"]).round(2)
    M[f"gap_{name}_matched"] = (M[a_] - M[f"{b_}_matched"]).round(2)
print("[3/7] neural threshold matched to the hybrid on the Ar-APT human anchor", flush=True)
print(M[["seed", "anchor_target", "thr_neural", "anchor_neu_at_thr"]].to_string(index=False), flush=True)
print("", flush=True)
print(M[["seed"] + [c for c in M.columns if c.startswith("gap_")]].to_string(index=False), flush=True)

In [ ]:
# ============================ 4/7  A · does any gap survive the matching? ============================
def boot(h_scores, n_scores, thr_n, label):
    h = (np.asarray(h_scores) >= 0.5).astype(float)
    n = (np.asarray(n_scores) >= thr_n).astype(float)
    if len(h) == 0: return None
    obs = 100.0*(h.mean() - n.mean())
    idx = RNG.integers(0, len(h), size=(N_BOOT, len(h)))
    d = 100.0*(h[idx].mean(1) - n[idx].mean(1))
    lo, hi = np.percentile(d, [2.5, 97.5])
    sig = "yes" if (lo > 0) or (hi < 0) else "no"
    print(f"[4/7] {label:48s} gap {obs:+7.2f}  95% CI [{lo:+7.2f}, {hi:+7.2f}]  significant {sig}", flush=True)
    return {"scope": label, "n": len(h), "gap": round(obs, 2),
            "lo": round(float(lo), 2), "hi": round(float(hi), 2), "significant": sig}

out = []
for seed in seeds:
    thr = THR[seed]
    s = S[S.seed == seed]; pol = s[s.task == "polish"]; hum = s[s.task == "humanize"]
    ap = A[(A.seed == seed) & (A.kind == "human_polished")]
    ai = A[(A.seed == seed) & (A.kind == "ai_generated")]
    for sub_h, sub_n, tag in ((pol.p_hybrid_gen, pol.p_neural_gen, "stress polish"),
                              (hum.p_hybrid_gen, hum.p_neural_gen, "stress humanize"),
                              (ap.p_hybrid,      ap.p_neural,      "Ar-APT polished"),
                              (ai.p_hybrid,      ai.p_neural,      "Ar-APT machine")):
        out.append(boot(sub_h, sub_n, 0.5, f"seed {seed} · {tag} · threshold 0.5"))
        out.append(boot(sub_h, sub_n, thr, f"seed {seed} · {tag} · matched threshold"))
    print("", flush=True)
B = pd.DataFrame([r for r in out if r])

In [ ]:
# ============================ 5/7  C · which articles produce the gap ============================
S = S.copy()
S["h"] = (S.p_hybrid_gen >= 0.5).astype(int)
S["n"] = (S.p_neural_gen >= 0.5).astype(int)
S["disagree"] = (S.h != S.n).astype(int)
print("[5/7] disagreements per seed and task", flush=True)
print(S.pivot_table(index="seed", columns="task", values="disagree", aggfunc="sum").to_string(), flush=True)
for task in ("polish", "humanize"):
    sets = {sd: set(S[(S.seed == sd) & (S.task == task) & (S.disagree == 1)].gen_id) for sd in seeds}
    union = set().union(*sets.values()); inter = set(sets[seeds[0]]).intersection(*sets.values())
    print(f"\n[5/7] {task}: union {len(union)} | in all three seeds {len(inter)} "
          f"| Jaccard(all) {len(inter)/max(len(union),1):.3f}", flush=True)
    for a_, b_ in itertools.combinations(seeds, 2):
        j = len(sets[a_] & sets[b_])/max(len(sets[a_] | sets[b_]), 1)
        print(f"[5/7]   seeds {a_} and {b_}: shared {len(sets[a_] & sets[b_]):3d} | Jaccard {j:.3f}", flush=True)

In [ ]:
# ============================ 6/7  C · which rewriter carries it ============================
if "model" in S.columns:
    piv = (S.groupby(["task", "model", "seed"])
             .apply(lambda s: 100.0*(s.h.mean() - s.n.mean()), include_groups=False)
             .rename("gap").reset_index())
    print("[6/7] gap by rewriting model and seed (hybrid minus neural)", flush=True)
    print(piv.pivot_table(index=["task", "model"], columns="seed", values="gap").round(2).to_string(), flush=True)
    stab = piv.groupby(["task", "model"]).gap.agg(["mean", "std", "min", "max"]).round(2)
    stab["sign_held"] = piv.groupby(["task", "model"]).gap.apply(lambda g: bool((g < 0).all() or (g > 0).all()))
    print("\n[6/7] stability across seeds, per rewriting model", flush=True)
    print(stab.to_string(), flush=True)
else:
    print("[6/7] no rewriting-model column, section skipped", flush=True)

In [ ]:
# ============================ 7/7  outputs and verdict ============================
D.to_parquet("/kaggle/working/nb22_threshold_drift.parquet", index=False)
M.to_parquet("/kaggle/working/nb22_threshold_matched.parquet", index=False)
B.to_parquet("/kaggle/working/nb22_threshold_bootstrap.parquet", index=False)
S[["gen_id", "seed", "task", "h", "n", "disagree"]].to_parquet(
    "/kaggle/working/nb22_disagreements.parquet", index=False)
print("[7/7] wrote nb22_threshold_drift, nb22_threshold_matched, nb22_threshold_bootstrap, nb22_disagreements", flush=True)

print("", flush=True)
print("[7/7] READ THIS AS FOLLOWS", flush=True)
for name in ("stress_FPR", "stress_TPR", "arapt_POL", "arapt_AI"):
    g0 = M[f"gap_{name}_05"]; gm = M[f"gap_{name}_matched"]
    shrunk = int((gm.abs() < g0.abs()/2).sum())
    print(f"[7/7]   {name:11s} mean gap at 0.5 {g0.mean():+6.2f} -> matched {gm.mean():+6.2f} "
          f"| seeds where it at least halves: {shrunk} of {len(M)} "
          f"| sign held at 0.5 {bool((g0<0).all() or (g0>0).all())}", flush=True)
print("[7/7]   a gap that shrinks toward zero once the operating points are matched is calibration;", flush=True)
print("[7/7]   a gap that survives is a difference in the decision rule itself.", flush=True)
print("", flush=True)
print(B.to_string(index=False), flush=True)